# Master Episode Pipeline
Enter a `NEW_EPISODE_ID` once — everything runs automatically and outputs one organized ZIP.

**Pipeline steps:**
1. FEMA NFIP Claims
2. USGS High-Water Marks
3. IFC Inundation Maps (KMZ + GeoPackage)
4. Sensor Time Series (IFC + USGS)
5. Interactive Folium Map
6. **Single ZIP export** — one prompt, one file, organized folders

In [ ]:
import ast, io, os, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

import folium
import geopandas as gpd
import fiona
import pandas as pd
import requests

fiona.drvsupport.supported_drivers['KML']    = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'
print('✓ Imports complete.')

In [ ]:
NOAA_EVENTS_FILE       = 'noaa_21-25_with_huc_08.csv'
IFC_SENSORS_FILE       = 'sensor_csv_huc08/ifis_stream_sensors_huc08.csv'
IFC_HYDROSTATIONS_FILE = 'sensor_csv_huc08/ifis_hydrostations_huc08.csv'
USGS_SENSORS_FILE      = 'sensor_csv_huc08/usgs_stream_sensors_huc08_with_foreign_id1.csv'
DEFAULT_EPISODE_ID     = '191899_0'

MAX_WORKERS         = 6
MAX_RETRIES         = 2
RETRY_DELAY_SECONDS = 4

COMMUNITY_IFIS_MAP = {
    'IOWA CITY':    598, 'JOHNSON':      598,
    'CEDAR RAPIDS': 501, 'LINN':         501,
    'CEDAR FALLS':  503, 'BLACK HAWK':   503,
    'DES MOINES':   502, 'POLK':         502,
}
IFC_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap',
}

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────────────────────

def normalize_huc(series):
    return series.fillna('').astype(str).str.replace(r'\.0$','',regex=True).str.strip().str.zfill(8)

def load_noaa_events(csv_path=NOAA_EVENTS_FILE):
    df = pd.read_csv(csv_path)
    df.columns = [c.upper().strip() for c in df.columns]
    df['FIPS_5']    = df['STATE_FIPS'].astype(str).str.zfill(2) + df['CZ_FIPS'].astype(str).str.zfill(3)
    df['HUC8_clean']= normalize_huc(df['HUC8'])
    df['BEGIN_DT']  = pd.to_datetime(df['BEGIN_DATE_TIME'])
    df['END_DT']    = pd.to_datetime(df['END_DATE_TIME'])
    return df

def get_lat_lon(row, lat_candidates, lon_candidates):
    """Robustly extracts lat/lon from a sensor row by trying multiple column names."""
    lat, lon = None, None
    for c in lat_candidates:
        v = row.get(c)
        if v is not None and pd.notna(v):
            try:
                lat = float(v)
                break
            except (ValueError, TypeError):
                pass
    for c in lon_candidates:
        v = row.get(c)
        if v is not None and pd.notna(v):
            try:
                lon = float(v)
                break
            except (ValueError, TypeError):
                pass
    return lat, lon

LAT_COLS = ['lat','latitude','LAT','LATITUDE','y','Y','lat_dd','dec_lat_va']
LON_COLS = ['lon','longitude','LON','LONGITUDE','x','X','lon_dd','lng','dec_long_va']

print('✓ Shared helpers defined.')

In [ ]:
# ── FEMA helpers ───────────────────────────────────────────────────────────────

def query_fema_claims_multi_fips(fips_list, start_date, end_date, date_buffer_days=14):
    base_url     = 'https://www.fema.gov/api/open/v2/FimaNfipClaims'
    search_start = (start_date - timedelta(days=2)).strftime('%Y-%m-%d')
    search_end   = (end_date   + timedelta(days=date_buffer_days)).strftime('%Y-%m-%d')
    fips_cond    = ' or '.join([f"countyCode eq '{f}'" for f in fips_list])
    fema_filter  = (f'({fips_cond}) and '
                    f'dateOfLoss ge {search_start}T00:00:00.000Z and '
                    f'dateOfLoss le {search_end}T23:59:59.000Z')
    try:
        r = requests.get(base_url, params={'$filter': fema_filter, '$top': 10000})
        r.raise_for_status()
        return pd.DataFrame(r.json().get('FimaNfipClaims', []))
    except Exception as e:
        print(f'Error querying OpenFEMA API: {e}')
        return pd.DataFrame()

print('✓ FEMA helpers defined.')

In [ ]:
# ── USGS HWM helpers ───────────────────────────────────────────────────────────

def query_usgs_hwms(min_lat, max_lat, min_lon, max_lon, start_date, end_date, date_buffer_days=30):
    try:
        r = requests.get('https://stn.wim.usgs.gov/STNServices/HWMs.json', timeout=15)
        r.raise_for_status()
        hwms = r.json()
        if not hwms:
            return pd.DataFrame()
        df_hwm  = pd.DataFrame(hwms)
        lat_col = 'latitude_dd'  if 'latitude_dd'  in df_hwm.columns else ('latitude'  if 'latitude'  in df_hwm.columns else None)
        lon_col = 'longitude_dd' if 'longitude_dd' in df_hwm.columns else ('longitude' if 'longitude' in df_hwm.columns else None)
        if not lat_col or not lon_col:
            print('Warning: coordinate columns not found in USGS response.')
            return pd.DataFrame()
        for col in ['flagDate','surveyDate','approvalDate']:
            if col in df_hwm.columns:
                df_hwm['date_parsed'] = pd.to_datetime(df_hwm[col], errors='coerce')
                break
        pad = 0.1
        mask_sp = ((df_hwm[lat_col] >= min_lat-pad) & (df_hwm[lat_col] <= max_lat+pad) &
                   (df_hwm[lon_col] >= min_lon-pad) & (df_hwm[lon_col] <= max_lon+pad))
        if 'date_parsed' in df_hwm.columns:
            s = start_date - timedelta(days=2)
            e = end_date   + timedelta(days=date_buffer_days)
            mask_t = (df_hwm['date_parsed'] >= s) & (df_hwm['date_parsed'] <= e)
            return df_hwm[mask_sp & mask_t]
        return df_hwm[mask_sp]
    except Exception as e:
        print(f'Error querying USGS STN API: {e}')
        return pd.DataFrame()

print('✓ USGS HWM helpers defined.')

In [ ]:
# ── IFC Inundation helpers ─────────────────────────────────────────────────────

def get_ifis_kmz_items(ifis_id):
    url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={ifis_id}'
    try:
        res  = requests.get(url, headers=IFC_HEADERS, timeout=10)
        res.raise_for_status()
        data = ast.literal_eval(res.text)
        items = []
        if len(data) > 2 and data[2] and data[2][0]:
            for mi in data[2][0][0]:
                items.append((mi[0], f'stage_{mi[1]}ft'))
        if len(data) > 2 and len(data[2]) > 1 and data[2][1]:
            for mi in data[2][1][0]:
                items.append((mi[0], f'flood_{mi[3]}yr'))
        return items
    except Exception as e:
        print(f'  [-] IFIS ID {ifis_id}: {e}')
        return []

def convert_kmz_bytes_to_geodataframe(kmz_bytes, extent_name):
    try:
        with zipfile.ZipFile(io.BytesIO(kmz_bytes)) as z:
            kml_files = [f for f in z.namelist() if f.endswith('.kml')]
            if not kml_files:
                return None
            kml_data = z.read(kml_files[0])
        gdf = gpd.read_file(io.BytesIO(kml_data), driver='KML')
        gdf['extent_name'] = extent_name
        if gdf.crs is None or gdf.crs.to_epsg() != 3418:
            gdf = gdf.to_crs(epsg=3418)
        gdf['area_acres'] = gdf.geometry.area / 4046.8564224
        return gdf
    except Exception as e:
        print(f'  [-] KMZ parse failed for {extent_name}: {e}')
        return None

print('✓ IFC inundation helpers defined.')

In [ ]:
# ── Sensor pipeline helpers ────────────────────────────────────────────────────

def resolve_hydrostation_id(row):
    for col in ['foreign_id1','ifc_id','station_id','hydro_id','ifis_id','ifc_id.1','Sensor_ID']:
        val = row.get(col)
        if pd.notna(val):
            try:
                n = int(float(val))
                if n > 4000: return str(n)
            except ValueError: pass
    raw = row.get('id')
    if pd.notna(raw):
        try: return str(int(float(raw)))
        except ValueError: pass
    return None

def fetch_nws_lid_data(nws_id, start_dt, end_dt):
    nws_id = str(nws_id).strip().upper()
    try:
        res = requests.get(f'https://api.water.noaa.gov/v1/gauges/{nws_id}/stageflow/observed',
                           headers={'User-Agent':'Mozilla/5.0','Accept':'application/json'}, timeout=12)
        if res.status_code == 200:
            records = []
            for obs in res.json().get('data',[]):
                t = pd.to_datetime(obs.get('validTime'))
                if t.tzinfo: t = t.tz_localize(None)
                if start_dt <= t <= end_dt:
                    records.append({'datetime':t,'parameter':'Stage / Observed','value':obs.get('primary'),'unit':obs.get('primaryUnit','ft')})
            if records:
                df = pd.DataFrame(records).sort_values('datetime')
                print(f'      ✅ {len(df)} records for NWS LID {nws_id}')
                return df
    except Exception: pass
    return pd.DataFrame()

def fetch_ifc_sensor_data(sensor_row, start_dt, end_dt):
    start_str = start_dt.strftime('%Y%m%d'); end_str = end_dt.strftime('%Y%m%d')
    candidates = [c for c in [sensor_row.get('id'), sensor_row.get('foreign_id')] if c is not None]
    for c in candidates:
        try:
            res = requests.get(f'https://hydroiowa.org/api/riversensor/{c}/data/{start_str}/{end_str}', timeout=15)
            if res.status_code == 200:
                observed = res.json().get('observed',[])
                if observed:
                    df = pd.DataFrame(observed)
                    if 'validTime' in df.columns:
                        df['validTime'] = pd.to_datetime(df['validTime'],format='mixed',errors='coerce')
                        df = df.dropna(subset=['validTime'])
                        if df['validTime'].dt.tz is not None: df['validTime'] = df['validTime'].dt.tz_localize(None)
                        df = df[(df['validTime']>=start_dt)&(df['validTime']<=end_dt)].copy()
                    if not df.empty:
                        print(f'      ✅ {len(df)} stage records for IFC sensor {c}')
                        return df
        except Exception: continue
    print(f"      ℹ️ No data for IFC sensor {sensor_row.get('foreign_id')}")
    return pd.DataFrame()

def fetch_ifc_hydrostation_data(station_row, start_dt, end_dt):
    start_str = start_dt.strftime('%Y%m%d'); end_str = end_dt.strftime('%Y%m%d')
    nid = resolve_hydrostation_id(station_row)
    name = station_row.get('foreign_id1')
    if not nid:
        print(f'      ❌ Skipping {name}: no numeric ID')
        return pd.DataFrame()
    hdrs = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    params = ['rain','wind','soil','well','groundwell','stage']
    all_frames = []
    for url in [f"https://hydroiowa.org/api/hydrostation/{nid}/data/{start_str}/{end_str}/?{'&'.join(params)}",
                 f'https://hydroiowa.org/api/hydrostation/{nid}/data/{start_str}/{end_str}']:
        if all_frames: break
        try:
            res = requests.get(url, headers=hdrs, timeout=15)
            if res.status_code == 200:
                obs = res.json().get('observed',{})
                if isinstance(obs, dict):
                    for k,v in obs.items():
                        if isinstance(v,list) and v:
                            p = pd.DataFrame(v); p['parameter_type']=k; all_frames.append(p)
                elif isinstance(obs,list) and obs:
                    all_frames.append(pd.DataFrame(obs))
        except Exception: pass
    if not all_frames:
        print(f'      ℹ️ No data for hydrostation {nid} ({name})')
        return pd.DataFrame()
    df = pd.concat(all_frames, ignore_index=True)
    if 'validTime' in df.columns:
        df['validTime'] = pd.to_datetime(df['validTime'],format='mixed',errors='coerce')
        df = df.dropna(subset=['validTime'])
        if df['validTime'].dt.tz is not None: df['validTime'] = df['validTime'].dt.tz_localize(None)
        df = df[(df['validTime']>=start_dt)&(df['validTime']<=end_dt)]
    if not df.empty: print(f'      ✅ {len(df)} records for hydrostation {nid} ({name})')
    else: print(f'      ℹ️ No records in range for hydrostation {nid} ({name})')
    return df

def fetch_usgs_sensor_data(usgs_row, start_dt, end_dt):
    candidates, nws_lids = [], []
    for key in ['foreign_id1','foreign_id','usgs_site_no_revised','usgs_site_no','id']:
        val = usgs_row.get(key)
        if pd.notna(val):
            cv = str(val).split('.')[0].strip().upper()
            if cv and cv != 'NAN' and cv not in candidates:
                candidates.append(cv)
                if not cv.isdigit(): nws_lids.append(cv)
    if not candidates:
        print('      ❌ Skipping USGS row: no identifier')
        return pd.DataFrame()
    strategies = []
    for sid in candidates:
        if sid.isdigit(): strategies += [('sites',sid.zfill(8)),('nwsLids',sid)]
        else:             strategies += [('nwsLids',sid),('sites',sid)]
    url = 'https://waterservices.usgs.gov/nwis/iv/'
    hdrs = {'User-Agent':'Mozilla/5.0 HydrologicalPipeline/2.0'}
    for param_type, qval in strategies:
        for pcd in ['00065','00060',None]:
            params = {'format':'json', param_type:qval,
                      'startDT':start_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
                      'endDT':  end_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')}
            if pcd: params['parameterCd'] = pcd
            try:
                res = requests.get(url, params=params, headers=hdrs, timeout=15)
                if res.status_code != 200: continue
                records = []
                for ts in res.json().get('value',{}).get('timeSeries',[]):
                    pname = ts.get('variable',{}).get('variableName','Unknown')
                    unit  = ts.get('variable',{}).get('unit',{}).get('unitCode','N/A')
                    for v in ts.get('values',[{}])[0].get('value',[]):
                        records.append({'datetime':v.get('dateTime'),'parameter':pname,'value':v.get('value'),'unit':unit})
                if records:
                    df = pd.DataFrame(records)
                    print(f'      ✅ {len(df)} records for USGS {qval}')
                    return df
            except Exception: continue
    for lid in nws_lids:
        df_n = fetch_nws_lid_data(lid, start_dt, end_dt)
        if df_n is not None and not df_n.empty: return df_n
    print(f"      ℹ️ No data for site {candidates[0]}")
    return pd.DataFrame()

def fetch_with_retry(fetch_fn, fetch_args):
    last_exc = None
    for attempt in range(1, MAX_RETRIES+1):
        try:
            df = fetch_fn(*fetch_args)
            if df is not None and not df.empty: return df
        except Exception as exc: last_exc = exc
        if attempt < MAX_RETRIES: time.sleep(RETRY_DELAY_SECONDS)
    if last_exc: raise last_exc
    return pd.DataFrame()

print('✓ Sensor helpers defined.')

In [ ]:
# ── Folium map builder ─────────────────────────────────────────────────────────

def fetch_huc8_geojson(huc8_code):
    """
    Fetches the actual HUC-8 polygon from the USGS NHD REST API as GeoJSON.
    Returns a dict (GeoJSON feature) or None.
    """
    url = 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query'
    params = {
        'where':        f"huc8='{huc8_code}'",
        'outFields':    'huc8,name',
        'f':            'geojson',
        'outSR':        '4326',
        'returnGeometry': 'true',
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        features = r.json().get('features', [])
        return features[0] if features else None
    except Exception as e:
        print(f'  ⚠️  Could not fetch HUC-8 polygon for {huc8_code}: {e}')
        return None


def build_episode_map(episode_id, episode_rows,
                      hwms_df=None,
                      ifc_sensors_df=None,
                      ifc_stations_df=None,
                      usgs_sensors_df=None):
    min_lat = episode_rows['BEGIN_LAT'].min()
    max_lat = episode_rows['END_LAT'].max()
    min_lon = episode_rows['BEGIN_LON'].min()
    max_lon = episode_rows['END_LON'].max()
    center  = [(min_lat+max_lat)/2, (min_lon+max_lon)/2]

    m = folium.Map(location=center, zoom_start=8, tiles='CartoDB positron')

    # 1. Episode bounding box
    folium.Rectangle(
        bounds=[[min_lat, min_lon],[max_lat, max_lon]],
        color='#1f78b4', weight=2.5, fill=True,
        fill_color='#1f78b4', fill_opacity=0.06,
        popup=f'NOAA Episode: {episode_id}', tooltip='Episode bounding box'
    ).add_to(m)

    # 2. IFC WMS inundation overlay
    folium.WmsTileLayer(
        url='https://ifis.iowafloodcenter.org/arcgis/services/IFIS/InundationMaps/MapServer/WMSServer',
        layers='0', name='IFC Inundation (WMS)', fmt='image/png',
        transparent=True, overlay=True, control=True,
        attr='Iowa Flood Center / IIHR'
    ).add_to(m)

    # 3. HUC-8 watersheds — fetched as real GeoJSON polygons from USGS REST API
    huc8_codes = [h for h in episode_rows['HUC8_clean'].dropna().unique() if h != '00000000']
    huc8_name_map = dict(zip(
        episode_rows.drop_duplicates('HUC8_clean')['HUC8_clean'],
        episode_rows.drop_duplicates('HUC8_clean')['NAME']
    ))
    palette = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
               '#42d4f4','#f032e6','#bfef45','#fabed4','#469990']

    if huc8_codes:
        print(f'  Fetching {len(huc8_codes)} HUC-8 polygon(s) from USGS REST API...')
        huc8_group = folium.FeatureGroup(name=f'HUC-8 Watersheds ({len(huc8_codes)})', show=True)
        for i, huc in enumerate(huc8_codes):
            colour  = palette[i % len(palette)]
            basin_name = huc8_name_map.get(huc, huc)
            feature = fetch_huc8_geojson(huc)
            if feature:
                folium.GeoJson(
                    feature,
                    name=f'HUC-8: {basin_name}',
                    style_function=lambda x, c=colour: {
                        'color': c, 'weight': 2.5,
                        'fillColor': c, 'fillOpacity': 0.12
                    },
                    tooltip=folium.GeoJsonTooltip(fields=['huc8','name'], aliases=['HUC-8','Basin'])
                ).add_to(huc8_group)
                print(f'    ✓ {basin_name} ({huc})')
            else:
                print(f'    ⚠️  No polygon returned for {huc}')
        huc8_group.add_to(m)

    # 4. USGS High-Water Marks
    if hwms_df is not None and not hwms_df.empty:
        lat_col = 'latitude_dd'  if 'latitude_dd'  in hwms_df.columns else 'latitude'
        lon_col = 'longitude_dd' if 'longitude_dd' in hwms_df.columns else 'longitude'
        hwm_group = folium.FeatureGroup(name=f'USGS High-Water Marks ({len(hwms_df)})', show=True)
        for _, row in hwms_df.iterrows():
            try:
                lat, lon = float(row[lat_col]), float(row[lon_col])
            except (ValueError, TypeError, KeyError):
                continue
            folium.CircleMarker(
                location=[lat, lon], radius=5,
                color='#e31a1c', fill=True, fill_color='#fb9a99', fill_opacity=0.85,
                popup=folium.Popup(
                    f"<b>USGS HWM</b><br>Type: {row.get('hwmTypeName','N/A')}<br>"
                    f"Elev: {row.get('elevFt','N/A')} ft<br>Quality: {row.get('hwmQualityName','N/A')}<br>"
                    f"ID: {row.get('hwmID','N/A')}", max_width=220),
                tooltip='USGS HWM'
            ).add_to(hwm_group)
        hwm_group.add_to(m)

    # 5. IFC River Sensors
    if ifc_sensors_df is not None and not ifc_sensors_df.empty:
        ifc_s_group = folium.FeatureGroup(name=f'IFC River Sensors ({len(ifc_sensors_df)})', show=True)
        plotted = 0
        for _, row in ifc_sensors_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#2ca02c', fill=True, fill_color='#98df8a', fill_opacity=0.9,
                popup=folium.Popup(
                    f"<b>IFC River Sensor</b><br>ID: {row.get('foreign_id', row.get('id','N/A'))}<br>"
                    f"HUC-8: {row.get('HUC8','N/A')}", max_width=200),
                tooltip='IFC River Sensor'
            ).add_to(ifc_s_group)
            plotted += 1
        ifc_s_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(ifc_sensors_df)} IFC river sensors')

    # 6. IFC Hydrostations
    if ifc_stations_df is not None and not ifc_stations_df.empty:
        ifc_h_group = folium.FeatureGroup(name=f'IFC Hydrostations ({len(ifc_stations_df)})', show=True)
        plotted = 0
        for _, row in ifc_stations_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#ff7f0e', fill=True, fill_color='#ffbb78', fill_opacity=0.9,
                popup=folium.Popup(
                    f"<b>IFC Hydrostation</b><br>ID: {row.get('foreign_id1', row.get('id','N/A'))}<br>"
                    f"HUC-8: {row.get('HUC8','N/A')}", max_width=200),
                tooltip='IFC Hydrostation'
            ).add_to(ifc_h_group)
            plotted += 1
        ifc_h_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(ifc_stations_df)} IFC hydrostations')

    # 7. USGS Gauges
    if usgs_sensors_df is not None and not usgs_sensors_df.empty:
        usgs_group = folium.FeatureGroup(name=f'USGS Gauges ({len(usgs_sensors_df)})', show=True)
        plotted = 0
        for _, row in usgs_sensors_df.iterrows():
            lat, lon = get_lat_lon(row, LAT_COLS, LON_COLS)
            if lat is None or lon is None: continue
            sid = str(row.get('foreign_id1', row.get('usgs_site_no_revised', row.get('id','N/A')))).split('.')[0]
            folium.CircleMarker(
                location=[lat, lon], radius=7,
                color='#9467bd', fill=True, fill_color='#c5b0d5', fill_opacity=0.9,
                popup=folium.Popup(
                    f"<b>USGS Gauge</b><br>Site: {sid}<br>HUC-8: {row.get('HUC8','N/A')}", max_width=200),
                tooltip='USGS Gauge'
            ).add_to(usgs_group)
            plotted += 1
        usgs_group.add_to(m)
        print(f'  ✓ Plotted {plotted}/{len(usgs_sensors_df)} USGS gauges')

    folium.LayerControl(collapsed=False).add_to(m)
    return m

print('✓ Folium map builder defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MASTER PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

df = load_noaa_events(NOAA_EVENTS_FILE)

# ── Episode selection ──────────────────────────────────────────────────────────
print('==================================================')
print(' NOAA Episode -> Master Pipeline')
print('==================================================')

episodes_summary = df.groupby('NEW_EPISODE_ID').agg(
    event_types =('EVENT_TYPE', lambda x: ', '.join(x.unique())),
    counties    =('CZ_NAME',   lambda x: ', '.join(x.unique())),
    event_count =('EVENT_ID',  'count')
).reset_index()

print('\nSample Episodes:\n')
print(episodes_summary[['NEW_EPISODE_ID','event_types','counties','event_count']].head(10).to_string(index=False))
print('\n--------------------------------------------------')

user_input = input(f"\nEnter NEW_EPISODE_ID (or Enter for '{DEFAULT_EPISODE_ID}'): ").strip()
if not user_input:
    user_input = DEFAULT_EPISODE_ID

episode_rows = df[df['NEW_EPISODE_ID'].astype(str) == str(user_input)]
if episode_rows.empty:
    print(f"Episode '{user_input}' not found.")
    raise SystemExit

# ── Metadata ───────────────────────────────────────────────────────────────────
fips_list    = episode_rows['FIPS_5'].unique().tolist()
county_names = episode_rows['CZ_NAME'].unique().tolist()
huc8_names   = episode_rows['NAME'].unique().tolist()
huc8_codes   = [h for h in episode_rows['HUC8_clean'].unique().tolist() if h != '00000000']
min_date     = episode_rows['BEGIN_DT'].min()
max_date     = episode_rows['END_DT'].max()
min_lat      = episode_rows['BEGIN_LAT'].min()
max_lat      = episode_rows['END_LAT'].max()
min_lon      = episode_rows['BEGIN_LON'].min()
max_lon      = episode_rows['END_LON'].max()
safe_id      = user_input.replace('/', '_')
start_time   = min_date - timedelta(hours=3)
end_time     = max_date + timedelta(hours=3)

print('\n--------------------------------------------------')
print('Selected Episode Summary:')
print(f' • Episode ID:    {user_input}')
print(f" • Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}")
print(f" • Counties ({len(fips_list)}): {', '.join(county_names)} (FIPS: {', '.join(fips_list)})")
print(f" • HUC-8 Basins:  {', '.join(huc8_names)}")
print(f' • Date Window:   {min_date.strftime("%Y-%m-%d %H:%M")} to {max_date.strftime("%Y-%m-%d %H:%M")}')
print(f' • Bounding Box:  Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]')
print('--------------------------------------------------\n')

# ── 1. FEMA Claims ─────────────────────────────────────────────────────────────
print('[1/4] Querying OpenFEMA NFIP Claims...')
claims_df = query_fema_claims_multi_fips(fips_list=fips_list, start_date=min_date, end_date=max_date)
if claims_df.empty:
    print('  No FEMA NFIP claims found.')
else:
    print(f'  Found {len(claims_df)} matching claim(s)!\n')
    cols_show = [c for c in ['dateOfLoss','countyCode','amountPaidOnBuildingClaim','amountPaidOnContentsClaim'] if c in claims_df.columns]
    print(claims_df[cols_show].head(10).to_string(index=False))
    if 'amountPaidOnBuildingClaim' in claims_df.columns:
        tot_b = claims_df['amountPaidOnBuildingClaim'].sum()
        tot_c = claims_df['amountPaidOnContentsClaim'].sum() if 'amountPaidOnContentsClaim' in claims_df.columns else 0
        print('\nFinancial Totals for Matching Episode Claims:')
        print(f' • Building Payouts: ${tot_b:,.2f}')
        print(f' • Contents Payouts: ${tot_c:,.2f}')
        print(f' • Total Payouts:    ${(tot_b+tot_c):,.2f}')

print('\n--------------------------------------------------')

# ── 2. USGS High-Water Marks ───────────────────────────────────────────────────
print('[2/4] Querying USGS STN High-Water Marks...')
print(f'  Bounding Box: Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]')
print(f'  Date Window:  {min_date.strftime("%Y-%m-%d")} to {max_date.strftime("%Y-%m-%d")}')
hwms_df = query_usgs_hwms(min_lat=min_lat, max_lat=max_lat, min_lon=min_lon, max_lon=max_lon,
                           start_date=min_date, end_date=max_date)
if hwms_df.empty:
    print('  No matching USGS High-Water Marks found.')
else:
    print(f'\n  Found {len(hwms_df)} matching USGS High-Water Mark(s)!\n')
    cols_hwm = [c for c in ['hwmID','eventName','latitude_dd','longitude_dd','elevFt','hwmQualityName','hwmTypeName'] if c in hwms_df.columns]
    print(hwms_df[cols_hwm].head(10).to_string(index=False))

print('\n--------------------------------------------------')

# ── 3. IFC Inundation Maps ─────────────────────────────────────────────────────
print('[3/4] Fetching IFC Inundation Maps...')
target_ifis_ids = set()
for _, row in episode_rows.iterrows():
    cz   = str(row.get('CZ_NAME','')).upper().strip()
    narr = str(row.get('EVENT_NARRATIVE','')).upper()
    for key, ifis_id in COMMUNITY_IFIS_MAP.items():
        if key in cz or key in narr:
            print(f"  [>] Matched '{key}' -> IFIS ID {ifis_id}")
            target_ifis_ids.add(ifis_id)
if not target_ifis_ids:
    print('  [*] No community matched — defaulting to Iowa City (598)')
    target_ifis_ids.add(598)

all_kmz_items = set()
for ifis_id in target_ifis_ids:
    all_kmz_items.update(get_ifis_kmz_items(ifis_id))

gdfs         = []
kmz_zip_buf  = None
gpkg_file    = None

if all_kmz_items:
    gpkg_file = f'episode_{safe_id}_inundation_layers.gpkg'
    print(f'  Fetching {len(all_kmz_items)} KMZ files...')
    kmz_zip_buf = io.BytesIO()
    with zipfile.ZipFile(kmz_zip_buf, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
        for kmz_url, extent_name in sorted(all_kmz_items, key=lambda x: x[1]):
            raw_fn  = kmz_url.split('/')[-1]
            desc_fn = f'{extent_name}_{raw_fn}'
            try:
                r = requests.get(kmz_url, timeout=15)
                if r.status_code == 200:
                    zf.writestr(desc_fn, r.content)
                    print(f'    ✓ {desc_fn}')
                    gdf = convert_kmz_bytes_to_geodataframe(r.content, extent_name)
                    if gdf is not None and not gdf.empty:
                        gdfs.append(gdf)
            except Exception as e:
                print(f'    [-] Error downloading {kmz_url}: {e}')
    if gdfs:
        combined_gdf = pd.concat(gdfs, ignore_index=True)
        drop_cols    = [c for c in ['Description','StyleMap','styleUrl'] if c in combined_gdf.columns]
        combined_gdf = combined_gdf.drop(columns=drop_cols)
        combined_gdf.to_file(gpkg_file, driver='GPKG')
        print(f'  [✓] GeoPackage saved: {gpkg_file} (EPSG:3418)')
else:
    print('  [-] No inundation KMZ files found.')

print('\n--------------------------------------------------')

# ── 4. Sensor matching ─────────────────────────────────────────────────────────
print('[4/4] Matching sensors to episode HUC-8 watersheds...')
ifc_sensors       = pd.read_csv(IFC_SENSORS_FILE)
ifc_hydrostations = pd.read_csv(IFC_HYDROSTATIONS_FILE)
usgs_sensors      = pd.read_csv(USGS_SENSORS_FILE)

matched_ifc_sensors  = ifc_sensors[normalize_huc(ifc_sensors['HUC8']).isin(huc8_codes)]
matched_ifc_stations = ifc_hydrostations[normalize_huc(ifc_hydrostations['HUC8']).isin(huc8_codes)]
matched_usgs_sensors = usgs_sensors[normalize_huc(usgs_sensors['HUC8']).isin(huc8_codes)]

print(f'  • IFC River Sensors:  {len(matched_ifc_sensors)}')
print(f'  • IFC Hydrostations:  {len(matched_ifc_stations)}')
print(f'  • USGS Sensors:       {len(matched_usgs_sensors)}')

print('\n--------------------------------------------------')

# ── Map ────────────────────────────────────────────────────────────────────────
print('\nBuilding interactive Folium map...')
m = build_episode_map(
    episode_id      = user_input,
    episode_rows    = episode_rows,
    hwms_df         = hwms_df          if not hwms_df.empty          else None,
    ifc_sensors_df  = matched_ifc_sensors  if not matched_ifc_sensors.empty  else None,
    ifc_stations_df = matched_ifc_stations if not matched_ifc_stations.empty else None,
    usgs_sensors_df = matched_usgs_sensors if not matched_usgs_sensors.empty else None,
)
map_file = f'episode_{safe_id}_map.html'
m.save(map_file)
print(f"  ✓ Map saved to '{map_file}'")

print('\n--------------------------------------------------')

# ── Single download prompt ─────────────────────────────────────────────────────
total_sensors = len(matched_ifc_sensors) + len(matched_ifc_stations) + len(matched_usgs_sensors)

print(f"""
Ready to export. Here is what will be included in the ZIP:
  tabular/   — FEMA claims CSV {'✓' if not claims_df.empty else '(no data)'}
              — USGS HWMs CSV  {'✓' if not hwms_df.empty  else '(no data)'}
  spatial/   — IFC inundation GeoPackage {'✓' if gpkg_file else '(no data)'}
             — IFC raw KMZ archive {'✓' if kmz_zip_buf else '(no data)'}
  sensors/   — IFC River Sensors ({len(matched_ifc_sensors)} sensors)
             — IFC Hydrostations ({len(matched_ifc_stations)} stations)
             — USGS Gauges ({len(matched_usgs_sensors)} gauges)
  maps/      — Interactive HTML map ✓
""")

do_download = input('Would you like to download all outputs as a single organized ZIP? (y/n): ').strip().lower()

if do_download not in ['y','yes']:
    print('Export skipped.')
else:
    master_zip = f'episode_{safe_id}_all_outputs.zip'
    print(f"\nBuilding '{master_zip}'...")

    with zipfile.ZipFile(master_zip, 'w', zipfile.ZIP_DEFLATED) as zf:

        # README
        readme = '\n'.join([
            f'Episode ID:    {user_input}',
            f"Event Types:   {', '.join(episode_rows['EVENT_TYPE'].unique())}",
            f"Counties:      {', '.join(county_names)}",
            f"HUC-8 Basins:  {', '.join(huc8_names)}",
            f'Date Window:   {min_date.strftime("%Y-%m-%d %H:%M")} to {max_date.strftime("%Y-%m-%d %H:%M")}',
            f'Bounding Box:  Lat [{min_lat:.3f}, {max_lat:.3f}], Lon [{min_lon:.3f}, {max_lon:.3f}]',
            '',
            'Folder structure:',
            '  tabular/  — FEMA claims CSV, USGS HWM CSV',
            '  spatial/  — IFC inundation GeoPackage (EPSG:3418), raw KMZ archive',
            '  sensors/  — time-series CSVs per sensor (subfolders by type)',
            '  maps/     — interactive HTML map (open in any browser)',
            '',
            f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        ])
        zf.writestr('README.txt', readme)

        # tabular/
        if not claims_df.empty:
            zf.writestr(f'tabular/fema_claims_episode_{safe_id}.csv', claims_df.to_csv(index=False))
            print('  ✓ tabular/fema_claims')
        if not hwms_df.empty:
            zf.writestr(f'tabular/usgs_hwms_episode_{safe_id}.csv', hwms_df.to_csv(index=False))
            print('  ✓ tabular/usgs_hwms')

        # spatial/
        if kmz_zip_buf:
            zf.writestr(f'spatial/episode_{safe_id}_inundation_maps.zip', kmz_zip_buf.getvalue())
            print('  ✓ spatial/inundation_maps.zip')
        if gpkg_file and os.path.exists(gpkg_file):
            zf.write(gpkg_file, f'spatial/{gpkg_file}')
            print(f'  ✓ spatial/{gpkg_file}')

        # maps/
        if os.path.exists(map_file):
            zf.write(map_file, f'maps/{map_file}')
            print(f'  ✓ maps/{map_file}')

        # sensors/ — download now, stream directly into ZIP
        if total_sensors > 0:
            print(f'\n  Downloading sensor time series ({total_sensors} sensors, {MAX_WORKERS} threads)...')
            tasks = []
            for _, row in matched_ifc_sensors.iterrows():
                code = str(row.get('foreign_id1', row.get('id','sensor')))
                tasks.append(('sensors/ifc_river', code, fetch_ifc_sensor_data, (row, start_time, end_time)))
            for _, row in matched_ifc_stations.iterrows():
                code = str(row.get('foreign_id1', row.get('id','station'))).replace(' ','_')
                tasks.append(('sensors/ifc_hydrostations', code, fetch_ifc_hydrostation_data, (row, start_time, end_time)))
            for _, row in matched_usgs_sensors.iterrows():
                code = str(row.get('foreign_id1', row.get('usgs_site_no_revised', row.get('foreign_id', row.get('id','usgs'))))).split('.')[0].strip().upper()
                tasks.append(('sensors/usgs', code, fetch_usgs_sensor_data, (row, start_time, end_time)))

            completed = 0
            success_counts = {'sensors/ifc_river':0,'sensors/ifc_hydrostations':0,'sensors/usgs':0}
            total_counts   = {'sensors/ifc_river':0,'sensors/ifc_hydrostations':0,'sensors/usgs':0}
            for folder,*_ in tasks: total_counts[folder] += 1

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                future_to_task = {
                    executor.submit(fetch_with_retry, fn, args): (folder, code)
                    for folder, code, fn, args in tasks
                }
                for future in as_completed(future_to_task):
                    folder, code = future_to_task[future]
                    completed += 1
                    try:
                        result_df = future.result()
                    except Exception as exc:
                        print(f'      ⚠️ Error {folder}/{code}: {exc}')
                        continue
                    if result_df is not None and not result_df.empty:
                        success_counts[folder] += 1
                        zf.writestr(f'{folder}/{code}.csv', result_df.to_csv(index=False).encode('utf-8'))
                    if completed % 10 == 0 or completed == len(tasks):
                        print(f'     ...{completed}/{len(tasks)} sensors processed')

            print('\n  📊 Sensor Collection Summary:')
            print(f"     • IFC River Sensors:  {success_counts['sensors/ifc_river']}/{total_counts['sensors/ifc_river']}")
            print(f"     • IFC Hydrostations:  {success_counts['sensors/ifc_hydrostations']}/{total_counts['sensors/ifc_hydrostations']}")
            print(f"     • USGS Sensors:       {success_counts['sensors/usgs']}/{total_counts['sensors/usgs']}")
            grand_s = sum(success_counts.values()); grand_t = sum(total_counts.values())
            print(f'     • TOTAL:              {grand_s}/{grand_t}')

    print(f"\n✓ ZIP saved: '{master_zip}'")
    print('  Structure:')
    with zipfile.ZipFile(master_zip,'r') as zf:
        for name in sorted(zf.namelist()):
            print(f'    {name}')

In [ ]:
# Render map inline in Jupyter
m